# Motivating Example: TTA 이득/손해 구간을 정답 없이 구분할 수 있는가

런타임 > 런타임 유형 변경 > GPU (T4 이상, ViT-L은 A100 권장)

In [ ]:
# 매번 최신 코드를 받는다 (data/, outputs/는 그대로 둔다)
import os
if not os.path.isdir('/content/repo'):
    !git clone -q -b 'hyunjin_expriments#2' https://github.com/hazigamja-capstone/Milestone3_motivatingExample.git /content/repo
else:
    !git -C /content/repo pull -q
%cd /content/repo
!pip -q install open_clip_torch gdown

## 1. 데이터: 공식 ImageNet-C

- 출처: HuggingFace `haideraltahan/wds_imagenetc` (공식 ImageNet-C를 webp로 담은 것, 토큰 불필요)
- 손상 6종 x severity 5, 손상마다 클래스당 10장 = 1만 장
- 손상 하나에 원본 약 7GB를 스트리밍으로 읽고 1만 장만 남긴다 (원본은 저장 안 함, 캐시는 손상당 1.5GB)
- 한 번 만든 캐시는 드라이브에 복사해 두고, 런타임이 바뀌면 드라이브에서 가져온다

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_CACHE = '/content/drive/MyDrive/motivating_cache/imagenet_c'
LOCAL = 'data/imagenet_c'
import os
os.makedirs(LOCAL, exist_ok=True)
if os.path.isdir(DRIVE_CACHE):
    !cp -n {DRIVE_CACHE}/*.npy {LOCAL}/
!python prepare_inc.py --out {LOCAL}
!mkdir -p {DRIVE_CACHE} && cp -n {LOCAL}/*.npy {DRIVE_CACHE}/
!ls -la {LOCAL}

## 2. 실행

- 결과는 `outputs/`에 실행마다 npz 하나씩
- 시간이 부족하면 `--n_per_domain 5000`

In [ ]:
!python run.py --model ViT-B-16 --method sar

In [ ]:
!python run.py --model ViT-B-16 --method tent

In [ ]:
!python run.py --model ViT-L-14 --method sar --grad_ckpt

In [ ]:
!python run.py --model ViT-L-14 --method eta --grad_ckpt

## 3. 분석

In [ ]:
!python analyze.py --seg 512
from IPython.display import Image
Image('outputs/scatter.png')

## 4. 결과 보관

Colab 런타임이 끊기면 outputs가 사라지니 드라이브로 복사해 둔다.

In [ ]:
!mkdir -p /content/drive/MyDrive/motivating_outputs && cp outputs/*.npz outputs/*.csv outputs/*.png /content/drive/MyDrive/motivating_outputs/